# TP14 — Prédire le Nutri-Score

**Objectif :** comparer des classifieurs A–E sans fuite de données, puis expliquer les erreurs observées.

Le tableau comparatif réunit les baselines et les modèles évalués sur les mêmes plis.

Données : Open Food Facts contributors — licence ODbL 1.0.

## Protocole

L’exécution utilise Python 3.12 et scikit-learn 1.9.1. Depuis la racine du dépôt, lance le notebook avec `uv run --group notebooks --with scikit-learn==1.9.1 jupyter lab`.

- Conserver le split fourni par le TP13 ; ne pas le recalculer.
- Ajuster imputation, standardisation, encodage et modèle dans chaque pli de train.
- Exclure nutriscore_score, code, product_name, brands, split et la cible des features.
- Réutiliser les mêmes plis stratifiés et les mêmes métriques pour chaque modèle.
- Garder le test à l’écart jusqu’au choix du modèle ; l’évaluer une seule fois.
- Écrire les résultats avant leur interprétation.

In [ ]:
import os
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix, f1_score, make_scorer
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

RANDOM_STATE = 42
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').is_file()), Path.cwd())
print(f'Racine : {ROOT.resolve()}')

## 1. Charger et contrôler les données

Le dépôt peut contenir l’échantillon local ou un export daté du TP13. Si nécessaire, définis NUTRISCOPE_DATA avec le chemin du Parquet.

In [34]:
candidats = [
    Path(os.environ['NUTRISCOPE_DATA']).expanduser() if os.environ.get('NUTRISCOPE_DATA') else None,
    ROOT / 'data' / 'nutriscope_ml_sample.parquet',
    ROOT / 'data' / 'tp13' / 'exports' / 'features_nutriscore_20261125.parquet',
    ROOT / 'data-nutriscope' / 'ml' / 'nutriscope_ml_sample.parquet',
]
DATA_PATH = next((p for p in candidats if p is not None and p.is_file()), None)
if DATA_PATH is None:
    raise FileNotFoundError('Parquet ML introuvable. Définis NUTRISCOPE_DATA.')
df = pd.read_parquet(DATA_PATH)
print(f'{DATA_PATH.resolve()} — {len(df):,} lignes × {len(df.columns)} colonnes')
display(df.head())

C:\Users\Administrateur\Documents\NutriScope\data\nutriscope_ml_sample.parquet — 30,041 lignes × 21 colonnes


,code,product_name,brands,pnns_groups_1,pnns_groups_2,nova_group,additives_n,energy-kcal_100g,fat_100g,saturated-fat_100g,...,sugars_100g,fiber_100g,proteins_100g,salt_100g,fruits-vegetables-legumes_100g,nutriscore_score,nutriscore_grade,completeness,last_modified,split
0,0000112302660,Soupe potiron et kiri,Liebig,Fruits and vegetables,Soups,4,4,38.300000,1.50000,1.000000,...,2.000000,0.800000,0.90000,0.740000,34.312500,3,c,0.6875,2026-03-14,train
1,0000112302705,velouté de tomates et pointe d'origan,Liebig,Fruits and vegetables,Soups,4,0,28.000000,0.50000,0.300000,...,2.400000,1.000000,0.70000,0.670000,72.977007,1,b,0.7625,2026-03-19,test
2,0000112302754,pur soup' mouliné de 5 légumes verts,liebig,Composite foods,One-dish meals,3,0,24.802376,0.16035,0.044483,...,0.738538,0.837462,0.81314,0.035146,23.242188,0,a,0.9750,2026-03-26,train
3,0000112307261,Velouté de 10 légumes,Liebig,Fruits and vegetables,Soups,4,0,43.000000,1.60000,0.600000,...,1.300000,0.800000,0.70000,0.670000,34.981336,3,c,0.9000,2026-03-26,train
4,0000300000004,Lentilles Brunes Bio,Kazidomi,Composite foods,One-dish meals,3,0,54.000000,0.30000,0.000000,...,0.000000,4.000000,4.00000,0.080000,100.000000,-7,a,0.8875,2026-03-17,test


In [35]:
TARGET, SPLIT = 'nutriscore_grade', 'split'
NEVER_FEATURES = {'nutriscore_score', 'code', 'product_name', 'brands', SPLIT, TARGET}
NUMERIC_FEATURES = [
    'energy-kcal_100g', 'fat_100g', 'saturated-fat_100g', 'carbohydrates_100g',
    'sugars_100g', 'proteins_100g', 'salt_100g', 'fiber_100g',
    'fruits-vegetables-legumes_100g', 'nova_group', 'additives_n',
]
CATEGORICAL_FEATURES = ['pnns_groups_1', 'pnns_groups_2']
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
missing = {TARGET, SPLIT, *FEATURES}.difference(df.columns)
assert not missing, f'Colonnes absentes : {sorted(missing)}'
assert set(df[TARGET].dropna().unique()) <= set('abcde'), 'Grade inattendu'
assert set(df[SPLIT].dropna().unique()) == {'train', 'test'}, 'Split train/test attendu'
display(df.groupby([SPLIT, TARGET], observed=True).size().unstack(fill_value=0))
display(df[FEATURES].isna().mean().sort_values(ascending=False).rename('part_manquante'))

nutriscore_grade,a,b,c,d,e
split,,,,,
test,797,629,1217,1602,1720
train,3233,2601,5018,6340,6884


fiber_100g                        0.468193
nova_group                        0.436171
fruits-vegetables-legumes_100g    0.416597
additives_n                       0.412370
energy-kcal_100g                  0.000000
sugars_100g                       0.000000
carbohydrates_100g                0.000000
saturated-fat_100g                0.000000
fat_100g                          0.000000
proteins_100g                     0.000000
salt_100g                         0.000000
pnns_groups_1                     0.000000
pnns_groups_2                     0.000000
Name: part_manquante, dtype: float64

### Lecture des données

L’échantillon contient 30 041 produits, dont 24 076 pour l’entraînement et 5 965 pour le test. Le notebook reprend le split figé de l’export. Sur le train, le grade e est le plus fréquent (6 884 produits) et le grade b le moins fréquent (2 601). Sur l’ensemble de l’échantillon, les variables les plus incomplètes sont les fibres (46,82 %), le groupe NOVA (43,62 %), la part de fruits, légumes et légumineuses (41,66 %) et le nombre d’additifs (41,24 %).

In [36]:
train_df = df.loc[df[SPLIT].eq('train')].copy()
# Le test reste dans df et n’est pas transformé avant le choix final.
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
folds = list(cv.split(X_train, y_train))  # mêmes indices pour chaque modèle
SCORING = {
    'f1_macro': make_scorer(f1_score, average='macro'),
    'accuracy': make_scorer(accuracy_score),
    'balanced_accuracy': make_scorer(balanced_accuracy_score),
}
print(f'Train : {len(X_train):,} lignes ; plis CV : {len(folds)}')

Train : 24,076 lignes ; plis CV : 5


## 2. Baselines

Les quatre références utilisent les mêmes plis stratifiés et les mêmes métriques que les modèles complets.

In [ ]:
baselines = {
    "Dummy majoritaire": DummyClassifier(strategy="most_frequent"),
    "Dummy stratifié": DummyClassifier(strategy="stratified", random_state=RANDOM_STATE),
}
for profondeur in (3, 8):
    baselines[f"Arbre (profondeur {profondeur})"] = Pipeline([
        ("imputation", SimpleImputer(strategy="median", add_indicator=True)),
        ("model", DecisionTreeClassifier(max_depth=profondeur, random_state=RANDOM_STATE)),
    ])

resultats_cv = {}
for nom, baseline in baselines.items():
    # Les arbres de référence utilisent les variables numériques brutes.
    donnees = X_train[NUMERIC_FEATURES] if isinstance(baseline, Pipeline) else X_train
    resultats_cv[nom] = cross_validate(
        baseline, donnees, y_train, cv=folds, scoring=SCORING, error_score="raise",
    )

predictions_arbre = cross_val_predict(
    baselines["Arbre (profondeur 3)"], X_train[NUMERIC_FEATURES], y_train, cv=folds,
)
print("Grades prédits par l’arbre de profondeur 3", sorted(set(predictions_arbre)))

La baseline majoritaire classe correctement 28,59 % des produits. Son F1 macro est de 0,089, car les quatre autres grades ont un F1 nul. L’accuracy équilibrée vaut 0,20. Sur les cinq plis, l’arbre de profondeur 3 prédit les grades c, d et e.

## 3. Pipeline et modèles

Le pipeline crée un écart énergétique signé, impute les numériques avec des indicateurs de manque, les standardise, puis encode les catégories avec regroupement sous 30 occurrences. L’écart énergétique est la variable dérivée retenue pour reproduire le modèle évalué.

Sur le premier pli, le préprocesseur produit 64 colonnes. Les médianes et les moyennes apprises correspondent aux statistiques des lignes d’entraînement. La moyenne de l’énergie standardisée est proche de zéro sur ces lignes et vaut 0,001185 sur la validation.

In [ ]:
def ajouter_ecart_energie(X):
    X = X.copy()
    energie_estimee = (
        4 * X["proteins_100g"]
        + 4 * X["carbohydrates_100g"]
        + 9 * X["fat_100g"]
    )
    X["ecart_energie_kcal_100g"] = X["energy-kcal_100g"] - energie_estimee
    return X


num_cols = [*NUMERIC_FEATURES, "ecart_energie_kcal_100g"]

preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("scaler", StandardScaler()),
        ]),
        num_cols,
    ),
    (
        "cat",
        OneHotEncoder(handle_unknown="ignore", min_frequency=30),
        CATEGORICAL_FEATURES,
    ),
])

modeles = {
    "Régression logistique": LogisticRegression(max_iter=1000),
    "Forêt aléatoire": RandomForestClassifier(random_state=RANDOM_STATE),
}

pipelines = {}

for nom, modele in modeles.items():
    pipeline = Pipeline([
        ("variables_derivees", FunctionTransformer(ajouter_ecart_energie)),
        ("preprocessor", clone(preprocessor)),
        ("model", modele),
    ])
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=folds,
        scoring=SCORING,
        error_score="raise",
    )

    pipelines[nom] = pipeline
    resultats_cv[nom] = scores

In [ ]:
# Contrôler le prétraitement sur le premier pli, sans ouvrir le test final.
indices_apprentissage, indices_validation = folds[0]
preparation = clone(pipelines["Régression logistique"][:-1])
X_apprentissage_pli = X_train.iloc[indices_apprentissage]
X_validation_pli = X_train.iloc[indices_validation]
preparation.fit(X_apprentissage_pli)

preparation_numerique = preparation.named_steps["preprocessor"].named_transformers_["num"]
imputation = preparation_numerique.named_steps["imputer"]
standardisation = preparation_numerique.named_steps["scaler"]
apprentissage_enrichi = ajouter_ecart_energie(X_apprentissage_pli)[num_cols]
imputees = imputation.transform(apprentissage_enrichi)
np.testing.assert_allclose(
    imputation.statistics_, apprentissage_enrichi.median().to_numpy(dtype=float),
)
np.testing.assert_allclose(standardisation.mean_, imputees.mean(axis=0))

train_transforme = preparation.transform(X_apprentissage_pli)
validation_transformee = preparation.transform(X_validation_pli)
moyenne_train = float(train_transforme[:, 0].mean())
moyenne_validation = float(validation_transformee[:, 0].mean())
assert abs(moyenne_train) < 1e-9
print(f"Moyenne de l’énergie standardisée, entraînement = {moyenne_train:.3g}")
print(f"Moyenne de l’énergie standardisée, validation = {moyenne_validation:.6f}")
print(f"Nombre de colonnes après préparation = {train_transforme.shape[1]}")

## 4. Comparaison et déséquilibre

Les effectifs et la pondération sont étudiés sur le train. Le tableau rassemble les résultats de validation croisée.

In [67]:
effectifs = y_train.value_counts()
display(effectifs)
print(f"Rapport majoritaire/minoritaire = {effectifs.max() / effectifs.min():.2f}")

nutriscore_grade
e    6884
d    6340
c    5018
a    3233
b    2601
Name: count, dtype: int64

Rapport majoritaire/minoritaire = 2.65


In [ ]:
foret_equilibree = clone(pipelines["Forêt aléatoire"])
foret_equilibree.set_params(model__class_weight="balanced")

scores_equilibres = cross_validate(
    foret_equilibree,
    X_train,
    y_train,
    cv=folds,
    scoring=SCORING,
    error_score="raise",
)
resultats_cv["Forêt aléatoire équilibrée"] = scores_equilibres

f1_reference = resultats_cv["Forêt aléatoire"]["test_f1_macro"]
f1_equilibre = scores_equilibres["test_f1_macro"]
ecarts = f1_equilibre - f1_reference

print("F1 macro moyen avec pondération", f1_equilibre.mean())
print("Écarts par pli", ecarts)
print("Écart moyen", ecarts.mean())

tableau_comparatif = pd.DataFrame.from_dict({
    nom: {
        "F1 macro moyen": scores["test_f1_macro"].mean(),
        "F1 macro écart-type": scores["test_f1_macro"].std(),
        "Accuracy moyenne": scores["test_accuracy"].mean(),
        "Accuracy équilibrée moyenne": scores["test_balanced_accuracy"].mean(),
        "Ajustement moyen (s)": scores["fit_time"].mean(),
        "Évaluation moyenne (s)": scores["score_time"].mean(),
    }
    for nom, scores in resultats_cv.items()
}, orient="index").rename_axis("Modèle")
display(tableau_comparatif.round(5))

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

predictions_cv = cross_val_predict(
    foret_equilibree, X_train, y_train, cv=folds
)

ConfusionMatrixDisplay.from_predictions(
    y_train,
    predictions_cv,
    labels=list("abcde"),
    normalize="true",
    values_format=".2f",
    cmap="Blues",
)
plt.show()

Le rapport entre les grades e et b est de 2,65. La pondération augmente le F1 macro de 0,85640 à 0,85860. Quatre plis progressent, mais le gain de 0,22 point dépend surtout du premier pli. La forêt pondérée est retenue sur le F1 macro moyen.

La matrice normalisée donne un rappel de 75 % pour b, avec 14 % des b prédits c et 10 % prédits a. Le grade e atteint 93 % de rappel. L’accuracy donne autant de poids à chaque produit, alors que le F1 macro donne autant de poids à chaque grade et intègre précision et rappel.

## 5. Fuite et importances

On ajoute volontairement le score numérique, puis on mesure son effet et on vérifie que le pipeline retenu l’exclut.

In [ ]:
X_train_avec_fuite = X_train.assign(
    nutriscore_score=train_df["nutriscore_score"]
)

assert X_train_avec_fuite.index.equals(y_train.index)
assert "nutriscore_score" not in X_train.columns

print(X_train.shape, X_train_avec_fuite.shape)

In [ ]:
foret_avec_fuite = clone(foret_equilibree)
preprocesseur_avec_fuite = foret_avec_fuite.named_steps["preprocessor"]

preprocesseur_avec_fuite.set_params(
    transformers=[
        (nom, transformateur, [*colonnes, "nutriscore_score"])
        if nom == "num"
        else (nom, transformateur, colonnes)
        for nom, transformateur, colonnes in preprocesseur_avec_fuite.transformers
    ]
)

scores_avec_fuite = cross_validate(
    foret_avec_fuite,
    X_train_avec_fuite,
    y_train,
    cv=folds,
    scoring=SCORING,
    error_score="raise",
)

f1_avec_fuite = scores_avec_fuite["test_f1_macro"]
f1_sans_fuite = scores_equilibres["test_f1_macro"]

print("F1 macro avec fuite par pli", f1_avec_fuite)
print(f"F1 macro moyen avec fuite = {f1_avec_fuite.mean():.5f}")
print(f"Écart moyen avec la référence = {(f1_avec_fuite - f1_sans_fuite).mean():+.5f}")

In [73]:
from sklearn.inspection import permutation_importance

indices_train, indices_validation = folds[0]

modele_pli = clone(foret_avec_fuite).fit(
    X_train_avec_fuite.iloc[indices_train],
    y_train.iloc[indices_train],
)

importances = permutation_importance(
    modele_pli,
    X_train_avec_fuite.iloc[indices_validation],
    y_train.iloc[indices_validation],
    scoring=SCORING["f1_macro"],
    n_repeats=5,
    random_state=RANDOM_STATE,
)

importance_moyenne = pd.Series(
    importances.importances_mean,
    index=X_train_avec_fuite.columns,
).sort_values(ascending=False)

display(importance_moyenne.head(5))

nutriscore_score      0.681649
pnns_groups_1         0.023144
proteins_100g         0.019022
salt_100g             0.006958
saturated-fat_100g    0.005260
dtype: float64

In [74]:
assert "nutriscore_score" not in X_train.columns

for nom, _, colonnes in foret_equilibree.named_steps["preprocessor"].transformers:
    assert "nutriscore_score" not in colonnes, f"Fuite dans la branche {nom}"

In [75]:
modele_sans_fuite = clone(foret_equilibree).fit(
    X_train.iloc[indices_train],
    y_train.iloc[indices_train],
)

importances_sans_fuite = permutation_importance(
    modele_sans_fuite,
    X_train.iloc[indices_validation],
    y_train.iloc[indices_validation],
    scoring=SCORING["f1_macro"],
    n_repeats=5,
    random_state=RANDOM_STATE,
)

classement_sans_fuite = pd.Series(
    importances_sans_fuite.importances_mean,
    index=X_train.columns,
).sort_values(ascending=False)

display(classement_sans_fuite.head(7))

salt_100g             0.227510
saturated-fat_100g    0.147383
sugars_100g           0.129457
proteins_100g         0.092220
fat_100g              0.050915
energy-kcal_100g      0.046083
pnns_groups_1         0.029314
dtype: float64

In [76]:
print(classement_sans_fuite.loc["additives_n"])

0.005704182589770945


In [77]:
foret_sans_additifs = clone(foret_equilibree)
preprocesseur = foret_sans_additifs.named_steps["preprocessor"]

preprocesseur.set_params(
    transformers=[
        (nom, transformateur, [col for col in colonnes if col != "additives_n"])
        if nom == "num"
        else (nom, transformateur, colonnes)
        for nom, transformateur, colonnes in preprocesseur.transformers
    ]
)

scores_sans_additifs = cross_validate(
    foret_sans_additifs, X_train, y_train,
    cv=folds, scoring=SCORING, error_score="raise",
)

ecarts = (
    scores_equilibres["test_f1_macro"]
    - scores_sans_additifs["test_f1_macro"]
)
print("Effet de l’ajout par pli", ecarts)
print(f"Effet moyen de l’ajout = {ecarts.mean():+.5f}")

Effet de l’ajout par pli [ 0.00377561 -0.00263703  0.00250199  0.00541368  0.00186433]
Effet moyen de l’ajout = +0.00218


L’ajout de nutriscore_score fait passer le F1 macro moyen de 0,85860 à 0,98661. Sa permutation sur le premier pli retire 0,68165 de F1 macro, contre 0,02314 pour la deuxième variable. Le grade étant dérivé du score numérique, ce gain révèle une fuite de cible. Les assertions contrôlent les entrées et les sélecteurs de la forêt retenue.

Sans fuite, le sel, les graisses saturées, les sucres, les protéines et l’énergie figurent parmi les sept premières importances. Les lipides totaux (qui participent aussi à l’écart énergétique) et la famille de produits apportent des indices indirects. La permutation de additives_n retire 0,00570 sur le premier pli. Son ajout à la forêt apporte 0,00218 en moyenne en validation croisée. Le nombre d’additifs déclarés est un témoin légitime dont l’effet est modeste et variable entre plis.

## 6. Test final et trois erreurs

In [78]:
test_df = df.loc[df[SPLIT].eq("test")].copy()
X_test = test_df[FEATURES]
y_test = test_df[TARGET]

modele_final = clone(foret_equilibree).fit(X_train, y_train)
predictions_test = modele_final.predict(X_test)

print(f"F1 macro test = {f1_score(y_test, predictions_test, average='macro'):.5f}")
print(f"Accuracy test = {accuracy_score(y_test, predictions_test):.5f}")
print(f"Accuracy équilibrée test = {balanced_accuracy_score(y_test, predictions_test):.5f}")

F1 macro test = 0.85775
Accuracy test = 0.87360
Accuracy équilibrée test = 0.85867


In [79]:
grades = {grade: rang for rang, grade in enumerate("abcde")}

erreurs = test_df[["code", "product_name", *NUMERIC_FEATURES]].copy()
erreurs["grade_reel"] = y_test
erreurs["grade_predit"] = predictions_test
erreurs["confiance"] = modele_final.predict_proba(X_test).max(axis=1)
erreurs["ecart_grade"] = (
    erreurs["grade_reel"].map(grades)
    - erreurs["grade_predit"].map(grades)
).abs()

erreurs = erreurs.loc[erreurs["ecart_grade"] > 0].sort_values(
    ["ecart_grade", "confiance"], ascending=False
)

display(erreurs[[
    "code", "product_name", "grade_reel", "grade_predit",
    "ecart_grade", "confiance",
]].head(15))

,code,product_name,grade_reel,grade_predit,ecart_grade,confiance
12151,3292590853020,2 boudins blancs au foie de canard,a,e,4,0.94
14692,3435079013717,Craquant rose framboise noir,a,e,4,0.92
5516,30182189,Préparation penne toscane épinards tomates,a,e,4,0.47
6265,3073190010880,Papillons Kilo Malin,d,a,3,0.96
29086,8710522647585,Maille vingt agr gin 6x1l,a,d,3,0.52
20192,3760109640312,Les Oublies de Nancy (chocolat ),a,d,3,0.51
22311,4016369521045,Élixir du suedois,e,b,3,0.50
25645,7312040090709,Absolut - Citron (lemon) Swedish Vodka 70cl Bo...,e,b,3,0.42
24940,5600826209561,Beurre Damandes,a,d,3,0.38
371,0064445520028,Tomates en dés sans sel,d,a,3,0.37


In [ ]:
codes_erreurs = ["3292590853020", "3073190010880", "7312040090709"]

colonnes = [
    "product_name",
    *NUMERIC_FEATURES,
    *CATEGORICAL_FEATURES,
    "nutriscore_score",
]

fiches_test = test_df.set_index(test_df["code"].astype(str))
display(fiches_test.loc[fiches_test.index.isin(codes_erreurs), colonnes].T)

### Évaluation sur le test

La forêt aléatoire avec `class_weight="balanced"` a été retenue sur la base
du F1 macro en validation croisée, puis entraînée sur l’ensemble du train.
Le test correspond aux lignes du split figé du TP13.

| Métrique | Résultat sur le test |
|---|---:|
| F1 macro | 0,85775 |
| Accuracy | 87,36 % |
| Accuracy équilibrée | 85,87 % |

Le F1 macro sur le test est proche de celui obtenu en validation croisée
(0,85860), avec un écart de −0,00085.

### Analyse de trois erreurs

**Boudins blancs au foie de canard (`3292590853020`) — `a → e`, confiance 94 %**

La fiche indique 310,9 kcal, 10,91 g de graisses saturées et 1,73 g de sel
pour 100 g. Les teneurs en sel et en graisses saturées rendent plausible
une prédiction défavorable, alors que le score enregistré est de 0.
L’hypothèse est une incohérence entre les nutriments et le score stocké.
Il faut comparer les valeurs à l’étiquette, vérifier les unités et vérifier
que le score et les nutriments proviennent de la même version de la fiche.

**Papillons Kilo Malin (`3073190010880`) — `d → a`, confiance 96 %**

La fiche indique 355 kcal, 70 g de glucides, 13 g de protéines, 0,30 g de
graisses saturées et 0,02 g de sel pour 100 g. Ces valeurs évoquent des
pâtes sèches. La faible teneur en sel et en graisses saturées pourrait
contribuer à la prédiction `a`, alors que le score enregistré est de 7.
Il faut vérifier si les nutriments et le score se rapportent au même état
du produit, sec ou préparé, puis identifier la version du barème appliquée.

**Absolut Citron (`7312040090709`) — `e → b`, confiance 42 %**

La fiche indique 242 kcal pour 100 g et 0 g de lipides, de glucides et de
protéines. L’alcool pourrait expliquer cette énergie. Le modèle reçoit la
catégorie « boissons alcoolisées » et l’écart énergétique calculé, tandis
que le degré d’alcool est absent de ses entrées. La confiance de 42 %
indique que la prédiction `b` est incertaine. Il faut vérifier si les
boissons alcoolisées font partie du périmètre cible et, si elles y restent,
évaluer les performances sur cette catégorie séparément.

Les grades de référence proviennent des fiches du jeu de test. Les
diagnostics ci-dessus sont des hypothèses à vérifier sur les fiches produit.

Données : © Open Food Facts contributors — ODbL.

## Conclusion

La forêt aléatoire pondérée obtient 0,85860 en F1 macro sur les cinq plis et 0,85775 sur le test, soit un écart de −0,00085. La distribution des grades rend l’accuracy insuffisante à elle seule. Le grade b reste le plus difficile à retrouver, malgré le léger gain moyen apporté par la pondération.

Le score numérique a été exclu après l’expérience de fuite. Les trois erreurs de test conduisent à vérifier la cohérence d’une fiche de charcuterie, l’état du produit et le barème pour les pâtes, puis le périmètre des boissons alcoolisées. Ces pistes restent des hypothèses.